In [0]:
def rename_columns(spark, df_spark, max_len_column=22):
    trunc_names = max_len_column - 4
    df_len = pd.DataFrame(df_spark.columns, columns=['variable'])
    df_len['length'] = df_len['variable'].str.len()

    # Truncar las columnas que tienen más de 25 caracteres
    df_len['temp_name'] = np.where(
        df_len['length'] <= max_len_column,
        df_len['variable'],
        df_len['variable'].str[:trunc_names]
    )

    # Verificar las columnas que tienen el mismo nombre pero con diferente case
    df_len['temp_name_lower'] = df_len['temp_name'].str.lower()
    df_len['n_repeat'] = df_len.groupby('temp_name_lower').cumcount()

    df_len['temp_len'] = df_len['temp_name'].str.len()
    df_len['flag_change'] = df_len['temp_len'] != df_len['length']

    # Verificar si hay columnas con el mismo nombre pero con diferente case
    df_len['has_duplicate'] = df_len.groupby('temp_name_lower')['temp_name_lower'].transform('count') > 1
    df_len['flag_change'] = df_len['flag_change'] | df_len['has_duplicate']

    # Renombrar las columnas
    df_len['new_name'] = np.where(
        df_len['flag_change'],
        df_len['temp_name'] + '_' + df_len['n_repeat'].astype(str).str.zfill(3),
        df_len['variable']
    )

    df_len['final_length'] = df_len['new_name'].str.len()
    df_new_names = spark.createDataFrame(df_len[['variable', 'new_name']])

    dict_rename = df_len[df_len['flag_change']].set_index('variable')['new_name'].to_dict()
    df_output = df_spark.withColumnsRenamed(dict_rename)

    print('Number of columns changed:', df_len['flag_change'].sum())

    return df_output, df_new_names

def decimals_to_double(spark, df_spark):
    # Identificar las columnas decimales
    decimal_cols = {f.name for f in df_spark.schema.fields if isinstance(f.dataType, DecimalType)}
    print(f"Converted {len(decimal_cols)} decimal column(s) to double.")

    # Cast solo las columnas decimales; preservar los nombres y el orden de las columnas originales
    return df_spark.select([
        F.col(f.name).cast(DoubleType()).alias(f.name) if f.name in decimal_cols else F.col(f.name)
        for f in df_spark.schema.fields
    ])

def replace_sentinels_with_null(spark, df, sentinels=None):
    """
    Reemplaza valores centinela por null en todas las columnas numéricas del DataFrame.

    Parámetros
    ----------
    df : pyspark.sql.DataFrame
        DataFrame de entrada.
    sentinels : iterable (opcional)
        Valores a reemplazar por null. Si es None, se usa el listado por defecto.

    Retorna
    -------
    pyspark.sql.DataFrame
        DataFrame con los valores centinela reemplazados por null en columnas numéricas.
    """
    if sentinels is None:
        sentinels = (
            1111111111, -1111111111,
            2222222222, -2222222222,
            3333333333, -3333333333,
            4444444444,
            5555555555,
            6666666666,
            7777777777,
            99999,
        )

    numeric_types = (
        ByteType, ShortType, IntegerType, LongType,
        FloatType, DoubleType, DecimalType
    )

    # Selecciona solo columnas numéricas por tipo
    numeric_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, numeric_types)]
    if not numeric_cols:
        return df  # No hay columnas numéricas; retorna igual

    # Un solo paso vectorizado sobre el subconjunto de columnas numéricas
    # None => null en Spark
    return df.na.replace(list(sentinels), None, subset=numeric_cols)